<a href="https://colab.research.google.com/github/fynoeb/2311537001-fayiamatullahazhara-semester2/blob/master/SpeechProcessing_Tugas4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# INSTALASI LIBRARY

!pip install openai-whisper transformers torch torchaudio librosa soundfile jiwer numpy scipy
!pip install accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 10.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 5.7 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=de249b9502003f105cf47486408a9f8fdcc65ab20be11877da783d6d22cc1ff5
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [ ]:
# IMPORT

import os
import time
import numpy as np
import librosa
import soundfile as sf
import torch
import warnings
warnings.filterwarnings("ignore")

from scipy.io import wavfile
from IPython.display import Audio, display

In [ ]:
# UPLOAD FILE AUDIO

from google.colab import files

print("Upload file audio bahasa Indonesia (.wav):")
uploaded_indo = files.upload()
indo_filename = list(uploaded_indo.keys())[0]

print("\nUpload file audio bahasa Inggris (.wav):")
uploaded_inggris = files.upload()
inggris_filename = list(uploaded_inggris.keys())[0]

print(f"\nFile Indonesia: {indo_filename}")
print(f"File Inggris : {inggris_filename}")

Upload file audio bahasa Indonesia (.wav):


Saving indofay.wav to indofay (1).wav

Upload file audio bahasa Inggris (.wav):


Saving inggrisfay.wav to inggrisfay (1).wav

File Indonesia: indofay (1).wav
File Inggris : inggrisfay (1).wav


In [ ]:
# RE-ENCODE AUDIO

import librosa
import soundfile as sf

def reencode_audio(input_path, output_path):
    audio, sr = librosa.load(input_path, sr=16000, mono=True)
    sf.write(output_path, audio, 16000, subtype='PCM_16')
    print(f"Re-encoded: {input_path} → {output_path} | Duration: {len(audio)/16000:.2f}s")
    return output_path

indo_clean_path    = reencode_audio(indo_filename,    "indo_clean.wav")
inggris_clean_path = reencode_audio(inggris_filename, "inggris_clean.wav")

Re-encoded: indofay (1).wav → indo_clean.wav | Duration: 76.58s
Re-encoded: inggrisfay (1).wav → inggris_clean.wav | Duration: 76.16s


In [ ]:
# PREPROCESSING: NOISE & SPEED

import numpy as np

def add_noise(audio, noise_factor=0.01):
    noise = np.random.randn(len(audio))
    return audio + noise_factor * noise

def change_speed(audio, speed_factor=1.5):
    return librosa.effects.time_stretch(audio, rate=speed_factor)

audio_indo,    _ = librosa.load("indo_clean.wav",    sr=16000)
audio_inggris, _ = librosa.load("inggris_clean.wav", sr=16000)

sf.write("indo_noisy.wav",    add_noise(audio_indo),          16000, subtype='PCM_16')
sf.write("inggris_noisy.wav", add_noise(audio_inggris),       16000, subtype='PCM_16')
sf.write("indo_fast.wav",     change_speed(audio_indo),       16000, subtype='PCM_16')
sf.write("inggris_fast.wav",  change_speed(audio_inggris),    16000, subtype='PCM_16')

print("Semua file audio siap.")

Semua file audio siap.


In [ ]:
# MODEL 1: WHISPER (OpenAI)

import whisper

whisper_model = whisper.load_model("base")

def transcribe_whisper(audio_path, language=None):
    start = time.time()
    if language:
        result = whisper_model.transcribe(audio_path, language=language)
    else:
        result = whisper_model.transcribe(audio_path)
    elapsed = time.time() - start
    return result["text"].strip(), elapsed

print("=== MODEL 1: WHISPER ===")
audio_files = {
    "Indo - Clean"   : ("indo_clean.wav", "id"),
    "Indo - Noisy"   : ("indo_noisy.wav", "id"),
    "Indo - Fast"    : ("indo_fast.wav",  "id"),
    "Inggris - Clean": ("inggris_clean.wav", "en"),
    "Inggris - Noisy": ("inggris_noisy.wav", "en"),
    "Inggris - Fast" : ("inggris_fast.wav",  "en"),
}

whisper_results = {}
for label, (path, lang) in audio_files.items():
    text, elapsed = transcribe_whisper(path, language=lang)
    whisper_results[label] = text
    print(f"\n[{label}] ({elapsed:.2f}s)\n{text}")

=== MODEL 1: WHISPER ===

[Indo - Clean] (24.30s)
Kalian ini saya mau pratikin cara satu menit transkripe wawancara informan selesai. Nah biasanya kan kalau transkrimenungan itu kita butuh seminggu, gue minggu ibakan lebih lama ya. Nah di antara berbagai aplikasi transkripe yang sekarang ada, saya tuh cocoknya pakai super AI. Karena dia bukan hanya bisa transkripe tapi bisa berbagai hal untuk mengeffectifkan penulisan skripsi. Oke ini tutorialnya ya. Buka website www.superAI.idl lalu login pakai akun Google pribadi kalian. Biasa subscribenya merabang banget nih. Cuma 49,000 kerbulan untuk bisa dapat access unlimited ke AI yang bisa buat transkripsi. Cuma di skonspullu deh. Ini kalian bakal dapat diskontam bahan 5,000 lagi. Jadi cuma 44,000 aja. Milihat. Semua AI penting ada di sini, mulai ada di Jakipiti, geminis sampai banyak future-feature lainnya. Oke lanjut. Nah kalian pilih aja future transkripe ideo. Maksudkan, Fire Recaman loan cara kalian. Dan tanpa menunggu lama. Tadah langsun

In [ ]:
!pip install stable-ts

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.1/189.1 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for stable-ts: filename=stable_ts-2.19.1-py3-none-any.whl size=162765 sha256=8e8d32ea9db32441666010374e1d3eaec7751c867d4e8de0fbc065aed51e1ffa
  Stored in directory: /root/.cache/pip/wheels/18/10/fe/b9c3ab284b29f6a379c58d57b7447055964d730f2f2ea7f6e4
Successfully built stable-ts


In [ ]:
# MODEL 2: WHISPER MEDIUM

import whisper
import time

whisper_medium = whisper.load_model("medium")

def transcribe_whisper_medium(audio_path, language=None):
    start = time.time()
    if language:
        result = whisper_medium.transcribe(audio_path, language=language)
    else:
        result = whisper_medium.transcribe(audio_path)
    elapsed = time.time() - start
    return result["text"].strip(), elapsed

print("=== MODEL 2: WHISPER MEDIUM ===")
whisper_medium_results = {}
for label, (path, lang) in audio_files.items():
    text, elapsed = transcribe_whisper_medium(path, language=lang)
    whisper_medium_results[label] = text
    print(f"\n[{label}] ({elapsed:.2f}s)\n{text}")

wav2vec_results = whisper_medium_results

100%|█████████████████████████████████████| 1.42G/1.42G [00:15<00:00, 97.2MiB/s]


=== MODEL 2: WHISPER MEDIUM ===

[Indo - Clean] (207.02s)
Kali ini saya mau praktikin cara 1 menit transcript wawancara informan selesai. Nah biasanya kan kalau transcript manual itu kita butuh seminggu, dua minggu, bahkan lebih lama ya. Nah di antara berbagai aplikasi transcript yang sekarang ada, saya tuh cocoknya pakai Super AI. Karena dia bukan hanya bisa transcript, tapi bisa berbagai hal untuk mengefektifkan penulisan skripsi. Oke ini tutorialnya ya. Buka website www.superai.id lalu login pakai akun Google pribadi kalian. Biasa subscribenya murah banget nih, cuma 49 ribu per bulan untuk bisa dapet akses unlimited ke AI yang bisa buat rencarin skripsi. Jangan lupa diskon 10 deh, ini kalian bakal dapet diskon tambahan 5 ribu lagi. Jadi cuma 44 ribu aja. Nih lihat, semua AI penting ada di sini, mulai dari chat GPT, Gemini, sampai banyak fitur-fitur lainnya. Oke lanjut, nah kalian pilih aja fitur Transcribe IDO, masukkan file rekaman wawancara kalian dan tanpa menunggu lama, tadaa la

In [ ]:
!pip install SpeechRecognition pydub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 31.1 MB/s eta 0:00:00


In [ ]:
# MODEL 3: GOOGLE SPEECH RECOGNITION

import speech_recognition as sr
import time

recognizer = sr.Recognizer()

def transcribe_google(audio_path, language="id-ID"):
    with sr.AudioFile(audio_path) as source:
        audio_data = recognizer.record(source)
    start = time.time()
    try:
        text = recognizer.recognize_google(audio_data, language=language)
    except sr.UnknownValueError:
        text = "[tidak dapat dikenali]"
    except sr.RequestError as e:
        text = f"[error: {e}]"
    elapsed = time.time() - start
    return text.strip(), elapsed

lang_map_google = {
    "Indo - Clean": "id-ID", "Indo - Noisy": "id-ID", "Indo - Fast": "id-ID",
    "Inggris - Clean": "en-US", "Inggris - Noisy": "en-US", "Inggris - Fast": "en-US",
}

print("=== MODEL 3: GOOGLE SPEECH RECOGNITION ===")
google_results = {}
for label, (path, _) in audio_files.items():
    lang = lang_map_google[label]
    text, elapsed = transcribe_google(path, language=lang)
    google_results[label] = text
    print(f"\n[{label}] ({elapsed:.2f}s)\n{text}")

wav2vec_results = google_results

=== MODEL 3: GOOGLE SPEECH RECOGNITION ===

[Indo - Clean] (8.60s)
kali ini saya mau praktekin cara 1 menit transkrip wawancara informan selesai Biasanya kan kalau transfer manual tuh kita butuh seminggu dua minggu bahkan lebih lama ya diantara berbagai aplikasi transkrip yang sekarang ada saya tuh cocoknya pakai super et karena dia bukan hanya bisa Translate tapi bisa berbagai hal untuk mengefektifkan penulisan skripsi Oke ini tutorialnya ya Buka website www super aido.id lalu login pake akun Google pribadi kalian biaya subscribe nya murah banget nih cuma 49 ribu perbulan Untuk bisa dapat akses unlimited ke Ai yang bisa buat lancarin skripsi jangan lupa Diskon 10 DM dapat Diskon tambahan 5000 lagi jadi cuma 44000 aja melihat semua penting ada di sini mulai dari CCTV Gemini sampai banyak fitur-fitur lainnya lanjut audio masukan file rekaman wawancara kalian dan tanpa menunggu lama dadah langsung jadi hasilnya nah kalian tinggal nambahin speakernya terus bisa langsung copy atau download